<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import sys
from IPython.display import display, Javascript

def restart_kernel():
    """Restart the Jupyter Notebook kernel to reflect changes in modules and packages."""
    display(Javascript("Jupyter.notebook.kernel.restart()"))
    print("Kernel is restarting...")

restart_kernel()

import numpy as np
import torch
import tensorly as tl
from tensorly.decomposition import parafac
import sparse
import pickle
import os

assert os.path.exists('sptensor.pkl'), 'No such file.'
with open('sptensor.pkl', 'rb') as f:
    data = pickle.load(f)
data = data[:, :, :, :12, :]
expected_shape = data.shape
n_components = 10

# -------------------------------
# 3. NumPy-based BPTF (Aaron's original implementation)
# -------------------------------
# Import Aaron's BPTF (which uses NumPy/sparse.COO); 
# ensure that the bptf package is in your PYTHONPATH.
from bptf import BPTF as BPTF
import bptf

# Create the same data as a NumPy array and a corresponding binary mask
data_np = data
mask_np = np.zeros(expected_shape, dtype=int)
mask_np[:, :, :, 3, :] = 1
mask_np = sparse.COO((mask_np.copy()).astype(np.int64))

# Instantiate and fit the NumPy-based BPTF model.
# Note: This version uses its own preprocess() function and can work with sparse.COO.
# model_np = BPTF(data_shape=data_np.shape, n_components=n_components)
# model_np.fit(data, mask=mask_np, max_iter=50, verbose=False, missing_val=1)

# Reconstruct using arithmetic expectation
# reconstruction_np = model_np.reconstruct(mask=None, fill_value=1, drop_diag=False, style='arithmetic')
# frobenius_diff_np = np.sqrt(np.sum((data.todense() - reconstruction_np)**2))
# print("NumPy BPTF reconstruction Frobenius norm difference:", frobenius_diff_np)

<IPython.core.display.Javascript object>

Kernel is restarting...


In [2]:
# -------------------------------
# 1. PyTorch-based BPTF (your version)
# -------------------------------
# Import your PyTorch-based BPTF model (adjust filename as needed)
from own_implementation import BPTF as BPTF_torch
tl.set_backend('pytorch')

device = 'cuda'
# Create a tensor from a Poisson distribution (counts) and a matching mask; ensure types match
data_torch = torch.tensor(data.todense(), dtype=torch.float64, device=device)
mask_torch = torch.tensor((1-mask_np.todense()).astype(np.int64), dtype=torch.float64, device=device)

# Instantiate and fit the PyTorch-based BPTF model
model_torch = BPTF_torch(data_shape=expected_shape, n_components=n_components, device=device)
model_torch.fit(data_torch, mask=mask_torch, max_iter=50, tol=1e-10, verbose=True)
reconstruction_torch = model_torch.reconstruct(mask=mask_torch, style='arithmetic')
frobenius_diff_torch = torch.norm(data_torch - reconstruction_torch, p='fro').item()
print("PyTorch BPTF reconstruction Frobenius norm difference:", frobenius_diff_torch)

ELBO = -730068.9548345318, change = 0.012233323014620613, time taken = 0.7648091316223145: 100%|██████████| 50/50 [00:32<00:00,  1.55it/s] 

PyTorch BPTF reconstruction Frobenius norm difference: 40280.46422684656


In [3]:
# -------------------------------
# 2. TensorLy CP Decomposition
# -------------------------------
# Use TensorLy's parafac for CP decomposition (same rank as n_components)
cp_decomp = parafac(data_torch, rank=n_components, n_iter_max=100, init='random')
reconstruction_cp = tl.cp_to_tensor(cp_decomp)
frobenius_diff_cp = torch.norm(data_torch - reconstruction_cp, p='fro').item()
print("TensorLy CP decomposition Frobenius norm difference:", frobenius_diff_cp)

TensorLy CP decomposition Frobenius norm difference: 28572.069031145085
